# Label-based alignment: Wasserstein on every section pair, FGW with an α sweep on the best one

For every cross pair (X1–X6), in 2D, with the organ labels as the cross cost (as in `label_alignment.ipynb`):
1. **Wasserstein OT on the labels** (`s-none_f-lab_<marginals>`) on every human × mouse section pair; each coupling is scored in both directions.
2. **The pick:** per direction, the section pair with the highest Wasserstein κ_bal. The label cost sees the labels κ_bal scores, so this is the section pair whose organs overlap best, not a method.
3. **FGW OT on the labels and space** (`s-euc{α}_f-lab_<marginals>`; `s-euc` for α = 0.5) on the picked section pairs only, once per α of `ALPHAS`: α is the weight of the GW term on the Euclidean distances within each section, 1 − α the weight of the label cost. α → 0 is the Wasserstein coupling, so the Wasserstein score is the start of every α curve.

The result is the **influence of the spatial term**: each score as a function of α, from Wasserstein (α = 0) to FGW at the largest α.

**Marginals** (`MARGINALS`): `balanced` (`bal`); `semi` (`semi{τ}`: the query side balanced, the target side relaxed by τ, so one solve per direction); `unbalanced` (`ub{τ}`: both sides relaxed by τ).

**Scores**, per section pair and direction (`a_stack`: mouse sections are the queries, the human sections the targets; `b_stack`: the reverse):
- **κ_bal**, the benchmark's metric (0 = chance, 1 = every organ to itself). Circular here: the cost sees the labels it is scored on.
- **gene r, held-out genes**: the mean Pearson r per gene between the query's expression and the target's expression transported through the coupling, minus the same with the target's spots shuffled. The genes are the pair's 500 held-out 1:1 orthologs.
- **gene r, all genes**: the same on every 1:1 ortholog present in all sections of the pair.

**Points per section** (`MAX_OBS`): each section's bins are Voronoi-merged into at most `MAX_OBS` tiles; `None` keeps every bin. κ_bal is scored against the full-resolution bins at any `MAX_OBS`; gene r on the tiles.

**Before the first run:** the sections, ortholog counts and pair features at this `MAX_OBS` must exist, on the data server: `python -m mgw.bench --set reduce.max_obs=<MAX_OBS> prepare --tier full --pairs <PAIRS> --seeds <SEED>` (`null` for every bin). Pairs without them are skipped, with the command to prepare them. Scores are saved after every pair to `<root>/diag/label_alignment_best/`, the α plots next to them.

**Time:** one Wasserstein solve per section pair (seconds at 5000 points), and `len(ALPHAS)` FGW solves for each of the (at most two) picked section pairs per cross pair (about 1 min each at 5000 points).

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = os.environ["MKL_NUM_THREADS"] = os.environ["OPENBLAS_NUM_THREADS"] = "8"
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"   # return JAX's GPU memory after each solve

import sys
sys.path.append("..")

In [ ]:
# ============================================================
# Run settings (papermill parameters)
# ============================================================
PAIRS = None                     # cross pairs of bench/configs/series.yaml (a = human, b = mouse); None: all, X1-X6
SEED = 0
MAX_OBS = 5000                   # points per section (Voronoi-merged bins); None: every bin
MARGINALS = "balanced"           # "balanced" | "semi" (query side balanced, target side TAU) | "unbalanced" (both TAU)
TAU = 0.7                        # marginal relaxation of "semi" and "unbalanced" (1 = balanced)
ALPHAS = (0.1, 0.3, 0.5, 0.7, 0.9)  # weights of the GW term of the FGW runs, each in (0, 1]; 1 - alpha on the labels
INNER_TOL = None                 # Sinkhorn threshold; None: bench.yaml's (1e-7); 1e-5 is 7-13x faster (doc, next steps)
CONFIG = "bench.yaml"            # benchmark config (bench/configs)
PIPELINE = "sec2sec_s-euc_f-euc_bal"  # the 2D pipeline whose data and FGW settings are used
ROOT = None                      # benchmark root; None: $MGW_BENCH_DIR or <data dir>/bench

In [ ]:
import dataclasses
import time

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from ott.geometry import geometry
from ott.problems.linear import linear_problem
from ott.solvers.linear import sinkhorn

from mgw import crossspecies as cs
from mgw import orthologs
from mgw.bench import config, metrics, prepare, series, solvers
from mgw.bench.pipelines.common import Task

if MARGINALS not in ("balanced", "semi", "unbalanced"):
    raise ValueError(f"MARGINALS must be balanced, semi or unbalanced, not {MARGINALS!r}")
if MARGINALS != "balanced" and not 0 < TAU < 1:
    raise ValueError(f"TAU must lie in (0, 1) for {MARGINALS} marginals, not {TAU}")
ALPHAS = tuple(sorted({float(a) for a in ALPHAS}))
if not ALPHAS or not all(0 < a <= 1 for a in ALPHAS):
    raise ValueError(f"ALPHAS must lie in (0, 1] (alpha = 0 is the Wasserstein run), not {ALPHAS}")
MARG = {"balanced": "bal", "semi": f"semi{TAU:g}", "unbalanced": f"ub{TAU:g}"}[MARGINALS]
W_RUN = f"s-none_f-lab_{MARG}"


def fgw_run(alpha):
    return f"{'s-euc' if alpha == 0.5 else f's-euc{alpha:g}'}_f-lab_{MARG}"


DIRS = ("a_stack", "b_stack")
DIR_NAME = {"a_stack": "mouse → human (a_stack)", "b_stack": "human → mouse (b_stack)"}
if MAX_OBS is not None and int(MAX_OBS) < 2:
    raise ValueError(f"MAX_OBS must be a number of points or None (every bin), not {MAX_OBS!r}")
RES = "all" if MAX_OBS is None else str(int(MAX_OBS))
MAX_OBS_SET = f"reduce.max_obs={'null' if MAX_OBS is None else int(MAX_OBS)}"

root = prepare.bench_root(ROOT)
cfg = config.load(CONFIG, [MAX_OBS_SET])
SERIES = series.load(cfg.series_path())
PAIRS = list(SERIES.pairs) if PAIRS is None else list(PAIRS)
unknown = [x for x in PAIRS if x not in SERIES.pairs]
if unknown:
    raise KeyError(f"unknown pairs {unknown}; series.yaml has {list(SERIES.pairs)}")
FH, SH = cfg.feature_hash(), cfg.section_hash()


def prepare_cmd(pids):
    return f"python -m mgw.bench --set {MAX_OBS_SET} prepare --tier full --pairs {' '.join(pids)} --seeds {SEED}"


def missing_inputs(pair):
    """What a pair lacks at this MAX_OBS: its features and the ortholog counts of its sections"""
    miss = [] if prepare.feature_file(root, pair.id, SEED, FH).exists() else ["features"]
    return miss + [s for e in (pair.a, pair.b) for s in e.sections
                   if not prepare.counts_file(root, s, SEED, SH).exists()]


SKIPPED = {pid: m for pid in PAIRS if (m := missing_inputs(SERIES.pairs[pid]))}
RUN_PAIRS = [pid for pid in PAIRS if pid not in SKIPPED]
OUT = root / "diag" / "label_alignment_best"
OUT.mkdir(parents=True, exist_ok=True)
STEM = f"seed{SEED}_n{RES}_{MARG}"
CSV = OUT / f"{STEM}_scores.csv"
print(f"pairs {', '.join(RUN_PAIRS) or '(none)'}; runs {W_RUN} on every section pair, "
      f"{', '.join(fgw_run(a) for a in ALPHAS)} on the picked ones\nroot {root}")
for pid in RUN_PAIRS:
    pr = SERIES.pairs[pid]
    print(f"  {pid}: {pr.a.name} (human, {len(pr.a.sections)} sections) <-> {pr.b.name} (mouse, "
          f"{len(pr.b.sections)} sections), {len(pr.a.sections) * len(pr.b.sections)} section pairs")
if SKIPPED:
    print(f"skipped, inputs missing at MAX_OBS = {MAX_OBS}: "
          + "; ".join(f"{pid} ({', '.join(m[:3])}{', ...' if len(m) > 3 else ''})" for pid, m in SKIPPED.items())
          + f"\nprepare them with: {prepare_cmd(list(SKIPPED))}")
if not RUN_PAIRS:
    raise FileNotFoundError("no pair has its inputs at this MAX_OBS; run the prepare command above first")


def gpu_gb(na, nb):
    """Rough peak GPU memory (GB) of one FGW solve in float32: both distance matrices and about four human x mouse
    matrices (label cost, coupling, linearized cost, a product)."""
    return 4 * (na * na + nb * nb + 4 * na * nb) / 1e9


dev = jax.devices()[0]
GPU_LIMIT = (dev.memory_stats() or {}).get("bytes_limit") if dev.platform == "gpu" else None
print(f"device {dev.device_kind}: {f'{GPU_LIMIT / 1e9:.0f} GB' if GPU_LIMIT else 'memory unknown'}")

## The costs and the solvers

As in `label_alignment.ipynb`:
- **Labels:** the cost between two spots is 0 for the same organ, 1 for different organs, and 0.5 when either is `Other` or `Excluded` (no evidence either way), averaged over the organ compositions of the two spots.
- **Space:** max-normalized Euclidean distances within each section (centred, RMS radius 1), as in `pipelines/real2d.py`.

`s-none_f-lab` uses entropic OT on the label cost alone (OTT's Sinkhorn); `s-euc{α}_f-lab` the benchmark's FGW solver (`solvers.fgw`). Both use the benchmark's ε and uniform masses. A balanced or unbalanced coupling (human × mouse) serves both directions; a semi-balanced one keeps the query side balanced, so each direction has its own solve.

In [ ]:
def label_cost(comp_a, comp_b):
    """Organ-label cross cost: 0 same organ, 1 different organs, 0.5 when a side is Other/Excluded."""
    Ka, Kb = metrics.row_normalize(comp_a)[:, metrics.SCORED], metrics.row_normalize(comp_b)[:, metrics.SCORED]
    return 1 - Ka @ Kb.T - 0.5 * (1 - np.outer(Ka.sum(1), Kb.sum(1)))


def ot_linear(M, tau, p, tau_b=None):
    """Entropic OT on the cross cost alone (unbalanced for tau < 1), uniform masses; tau_b: the target side's
    relaxation, if not tau (tau = 1, tau_b < 1: semi-relaxed, the query side balanced)."""
    t0 = time.perf_counter()
    geom = geometry.Geometry(cost_matrix=jnp.asarray(M, dtype=jnp.float32), epsilon=p.epsilon)
    out = sinkhorn.Sinkhorn(max_iterations=p.inner_maxit, threshold=p.inner_tol)(
        linear_problem.LinearProblem(geom, tau_a=tau, tau_b=tau if tau_b is None else tau_b))
    return np.asarray(out.matrix, dtype=np.float64), dict(converged=bool(out.converged),
                                                          seconds=time.perf_counter() - t0)


def taus(direction):
    """(tau_a, tau_b) of the human x mouse coupling that serves `direction`; semi keeps the query side balanced."""
    if MARGINALS == "balanced":
        return 1.0, 1.0
    if MARGINALS == "unbalanced":
        return TAU, TAU
    return (1.0, TAU) if direction == "b_stack" else (TAU, 1.0)


def dist(task, s):
    """Max-normalized Euclidean distances within a section, float32 (the solver's precision)."""
    return solvers.dist_norm(task.xy(s)).astype(np.float32)


def fgw_params(task):
    return task.fgw if INNER_TOL is None else dataclasses.replace(task.fgw, inner_tol=INNER_TOL)


def solve_fgw(Ml, Ca, Cb, alpha, tau_a, tau_b, p):
    return solvers.fgw(None, None, Ml, dataclasses.replace(p, alpha=alpha), Ca=Ca, Cb=Cb, tau_a=tau_a, tau_b=tau_b)


# the label cost on toy spots: heart, heart, brain, Other
toy = metrics.onehot(["Heart", "Heart", "Brain", "Other"])
assert np.allclose(label_cost(toy, toy), [[0, 0, 1, .5], [0, 0, 1, .5], [1, 1, 0, .5], [.5, .5, .5, .5]])
# Wasserstein: balanced marginals are uniform; semi-relaxed keeps the query's
p_toy = solvers.FGWParams.from_mapping(cfg.raw["fgw"])
Mt = np.random.default_rng(0).random((30, 20))
Pt, _ = ot_linear(Mt, 1.0, p_toy)
assert np.allclose(Pt.sum(1), 1 / 30, atol=1e-4) and np.allclose(Pt.sum(0), 1 / 20, atol=1e-4)
Pt, _ = ot_linear(Mt, 1.0, p_toy, tau_b=0.7)
assert np.allclose(Pt.sum(1), 1 / 30, atol=1e-4)
print(f"toy checks passed; marginals {MARGINALS}: (tau_a, tau_b) = {taus('b_stack')} for b_stack (human query), "
      f"{taus('a_stack')} for a_stack (mouse query)")

## All genes

The log-normalized counts of every section (`prepare.load_counts`, 1:1 orthologs only), with mouse genes renamed to their human orthologs, restricted to the genes present in every section of the pair (without mitochondrial genes, as for the held-out genes). Loaded per pair.

In [ ]:
ortho = orthologs.load_one2one()
m2h = dict(zip(ortho.mouse, ortho.human))


def ortholog_counts(s):
    """(tiles x genes CSR, human gene symbols) of a section's 1:1 orthologs."""
    a = prepare.load_counts(root, s.name, SEED, SH)
    keep = a.var_names.isin(set(ortho.human) if s.dataset == "HESTA" else set(m2h))
    genes = a.var_names[keep] if s.dataset == "HESTA" else a.var_names[keep].map(m2h)
    return sp.csr_matrix(a.X[:, np.flatnonzero(keep)]), pd.Index(genes)


def load_genes(secs):
    """{section: tiles x genes CSR} on the genes present in every section of the pair, without MT- genes."""
    raw = {n: ortholog_counts(s) for n, s in secs.items()}
    genes = pd.Index(sorted(set.intersection(*(set(g) for _, g in raw.values()))))
    genes = genes[~genes.str.upper().str.startswith("MT-")]
    G = {n: X[:, g.get_indexer(genes)].tocsr() for n, (X, g) in raw.items()}
    assert all(G[n].shape[0] == s.n for n, s in secs.items()), "counts and sections disagree on the tiles"
    return G, len(genes)


def coexpression_all(Pq, Yq, Yt, seed=SEED, chunk=2000):
    """cs.coexpression in gene chunks: per gene, Pearson r over the query's spots between Yq and Yt transported
    through Pq, and the same with Yt's spots shuffled (the same permutation). Yq, Yt dense float32: the transport is
    a BLAS product (a sparse one runs single-threaded, ~20x slower at these sizes); r is computed in float64."""
    Pn = np.asarray(Pq, dtype=np.float32)
    Pn = Pn / (Pn.sum(1, keepdims=True) + 1e-30)
    perm = np.random.default_rng(seed).permutation(len(Yt))
    r, r0 = [], []
    for j in range(0, Yq.shape[1], chunk):
        cols = slice(j, j + chunk)
        Yqc, Ytc = Yq[:, cols].astype(np.float64), Yt[:, cols]
        r.append(cs.colwise_r(Yqc, (Pn @ Ytc).astype(np.float64)))
        r0.append(cs.colwise_r(Yqc, (Pn @ Ytc[perm]).astype(np.float64)))
    return np.concatenate(r), np.concatenate(r0)


# the chunked version is cs.coexpression
rng = np.random.default_rng(1)
Yq, Yt = rng.random((120, 300), dtype=np.float32), rng.random((90, 300), dtype=np.float32)
Pt = rng.random((120, 90))
ra, ra0 = coexpression_all(Pt, Yq, Yt, chunk=97)
rb, rb0 = cs.coexpression(Pt, Yq, Yt, seed=SEED)
assert np.allclose(ra, rb, atol=1e-5, equal_nan=True) and np.allclose(ra0, rb0, atol=1e-5, equal_nan=True)
del Yq, Yt, Pt

## Solve and score

Per pair: Wasserstein on every human × mouse section pair, then the pick, then the FGW α sweep on the picked section pairs. `partner`: the target is the query's partner section (the target section closest in laterality L, as in `section_cost_ablation.ipynb`). A section pair picked by both directions is solved once per α under balanced or unbalanced marginals; its distance matrices are built once for all α.

In [ ]:
def partner(q, targets):
    ok = {n: t.L for n, t in targets.items() if np.isfinite(t.L)}
    return min(ok, key=lambda n: abs(ok[n] - q.L)) if np.isfinite(q.L) and ok else None


def score_row(task, run, alpha, d, h, m, P, info, Gd, PARTNER):
    """One scored row: the human x mouse coupling P in direction d (Gd: dense counts of both sections)."""
    H, Mo = task.data["a"], task.data["b"]
    q, t, Pq = (H[h], Mo[m], P) if d == "b_stack" else (Mo[m], H[h], P.T)
    r, r0 = cs.coexpression(Pq, q.Y, t.Y, seed=SEED)
    ra, ra0 = coexpression_all(Pq, Gd[q.name], Gd[t.name])
    return dict(pair=task.pair.id, run=run, alpha=alpha, direction=d, query=q.name, target=t.name, human=h, mouse=m,
                partner=PARTNER[d, q.name] == t.name,
                kappa_bal=task.score(d, q, t.name, t.comp, Pq, keep=False)["kappa_bal"],
                gene_r_heldout=np.nanmean(r) - np.nanmean(r0), r_heldout=np.nanmean(r),
                gene_r_all=np.nanmean(ra) - np.nanmean(ra0), r_all=np.nanmean(ra),
                matched=Pq.sum(), converged=bool(info.get("converged")), seconds=info.get("seconds"))


def check(P, what):
    if not np.isfinite(P).all():
        raise FloatingPointError(f"non-finite coupling: {what}")


def run_wasserstein(task, G, PARTNER, p):
    """Wasserstein on every human x mouse section pair, scored in both directions."""
    H, Mo = task.data["a"], task.data["b"]
    rows = []
    for h in H:
        Gh = G[h].toarray()                      # dense expression for the BLAS products of coexpression_all
        for m in Mo:
            t0 = time.perf_counter()
            Gd = {h: Gh, m: G[m].toarray()}
            Ml = label_cost(H[h].comp, Mo[m].comp)
            sols = {}
            for d in DIRS:
                tt = taus(d)
                if tt not in sols:
                    sols[tt] = ot_linear(Ml, tt[0], p, tau_b=tt[1])
                    check(sols[tt][0], f"{W_RUN} {tt}, {h} x {m}")
                rows.append(score_row(task, W_RUN, np.nan, d, h, m, *sols[tt], Gd, PARTNER))
            print(f"  W  {h} ({H[h].n}) x {m} ({Mo[m].n}): {time.perf_counter() - t0:.0f} s, kappa_bal "
                  + ", ".join(f"{r['direction']} {r['kappa_bal']:.3f}" for r in rows[-len(DIRS):]))
    return pd.DataFrame(rows)


def pick(w):
    """Per direction, the (human, mouse) section pair with the highest Wasserstein kappa_bal (the first on ties)."""
    return {d: tuple(w.loc[w[w.direction == d].kappa_bal.idxmax(), ["human", "mouse"]]) for d in DIRS}


def run_fgw_sweep(task, G, PARTNER, picks, p):
    """FGW at every alpha on each direction's picked section pair, scored in that direction."""
    H, Mo = task.data["a"], task.data["b"]
    rows = []
    for hm in dict.fromkeys(picks.values()):     # each picked section pair once, in direction order
        h, m = hm
        dirs = [d for d in DIRS if picks[d] == hm]
        print(f"  FGW {h} ({H[h].n}) x {m} ({Mo[m].n}) for {', '.join(dirs)}: "
              f"~{gpu_gb(H[h].n, Mo[m].n):.1f} GB on the GPU (estimate)")
        if GPU_LIMIT and gpu_gb(H[h].n, Mo[m].n) > 0.9 * GPU_LIMIT / 1e9:
            print("  WARNING: this pair may not fit on the GPU; lower MAX_OBS or use a GPU with more memory")
        Gd = {h: G[h].toarray(), m: G[m].toarray()}
        Ml, Dh, Dm = label_cost(H[h].comp, Mo[m].comp), dist(task, H[h]), dist(task, Mo[m])
        for alpha in ALPHAS:
            sols = {}
            for d in dirs:
                tt = taus(d)
                if tt not in sols:
                    sols[tt] = solve_fgw(Ml, Dh, Dm, alpha, *tt, p)
                    check(sols[tt][0], f"{fgw_run(alpha)} {tt}, {h} x {m}")
                rows.append(score_row(task, fgw_run(alpha), alpha, d, h, m, *sols[tt], Gd, PARTNER))
                r = rows[-1]
                print(f"    alpha {alpha:g} {d}: {sols[tt][1]['seconds']:.0f} s, kappa_bal {r['kappa_bal']:.3f}, "
                      f"gene r held-out {r['gene_r_heldout']:.3f}, all {r['gene_r_all']:.3f}")
            del sols
        del Gd, Ml, Dh, Dm
    return pd.DataFrame(rows)


parts, SECTIONS, PICKS = [], {}, {}
for pid in RUN_PAIRS:
    t0 = time.perf_counter()
    task = Task(cfg, PIPELINE, SERIES.pairs[pid], SEED, root)
    p = fgw_params(task)
    H, Mo = task.data["a"], task.data["b"]
    SECTIONS[pid] = (list(H), list(Mo))
    PARTNER = {**{("b_stack", q): partner(H[q], Mo) for q in H}, **{("a_stack", q): partner(Mo[q], H) for q in Mo}}
    G, n_genes = load_genes({**H, **Mo})
    print(f"{pid}: {len(H)} human x {len(Mo)} mouse sections, points human {[s.n for s in H.values()]}, "
          f"mouse {[s.n for s in Mo.values()]}; {n_genes} genes in all sections")
    w = run_wasserstein(task, G, PARTNER, p)
    PICKS[pid] = pick(w)
    w["best"] = [PICKS[pid][r.direction] == (r.human, r.mouse) for r in w.itertuples()]
    for d, (h, m) in PICKS[pid].items():
        r = w[w.best & (w.direction == d)].iloc[0]
        print(f"  pick {d}: {r.query} -> {r.target} (partner: {r.partner}), Wasserstein kappa_bal {r.kappa_bal:.3f}")
    f = run_fgw_sweep(task, G, PARTNER, PICKS[pid], p)
    f["best"] = True
    parts += [w, f]
    pd.concat(parts, ignore_index=True).to_csv(CSV, index=False)    # saved after every pair
    print(f"{pid} done in {time.perf_counter() - t0:.0f} s\n")
    del task, G, H, Mo, w, f

scores = pd.concat(parts, ignore_index=True)
print(f"solver settings: {p}\nscores saved to {CSV}")
if not scores.converged.all():
    print("not converged:", scores.loc[~scores.converged, ["pair", "run", "human", "mouse"]].drop_duplicates()
          .to_string(index=False))

## The pick: Wasserstein κ_bal of every section pair

One row of panels per cross pair; rows are the human sections and columns the mouse sections, in both directions. Solid outline: the picked section pair of that direction; dashed: the query's partner section. One colour scale for all panels.

In [ ]:
HUMAN, MOUSE = "#2a78d6", "#eb6834"               # categorical slots 1 and 2: the query's species
INK, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
SEQ = LinearSegmentedColormap.from_list("blue", ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95",
                                                 "#0d366b"])
SEQ.set_bad(SURFACE)
METRICS = {"kappa_bal": "κ_bal", "gene_r_heldout": "gene r, held-out genes", "gene_r_all": "gene r, all genes"}
W = scores[scores.run == W_RUN]
F = scores[scores.run != W_RUN]

lo, hi = W.kappa_bal.min(), W.kappa_bal.max()
mid = lo + 0.55 * (hi - lo)
heights = [len(SECTIONS[pid][0]) for pid in RUN_PAIRS]
ncol_max = max(len(SECTIONS[pid][1]) for pid in RUN_PAIRS)
fig, axes = plt.subplots(len(RUN_PAIRS), len(DIRS), squeeze=False, layout="constrained",
                         figsize=(len(DIRS) * (0.85 * ncol_max + 2), sum(0.42 * n + 1.3 for n in heights)),
                         gridspec_kw=dict(height_ratios=[0.42 * n + 1.3 for n in heights]))
for i, pid in enumerate(RUN_PAIRS):
    hs, ms = SECTIONS[pid]
    for j, d in enumerate(DIRS):
        ax = axes[i, j]
        s = W[(W.pair == pid) & (W.direction == d)]
        v = s.pivot_table(index="human", columns="mouse", values="kappa_bal").reindex(index=hs, columns=ms).to_numpy()
        im = ax.imshow(np.ma.masked_invalid(v), cmap=SEQ, vmin=lo, vmax=hi, aspect="auto")
        for (y, x), val in np.ndenumerate(v):
            if np.isfinite(val):
                ax.text(x, y, f"{val:.2f}", ha="center", va="center", fontsize=7, color=INK if val < mid else SURFACE)
        for _, r in s[s.partner].iterrows():
            ax.add_patch(Rectangle((ms.index(r.mouse) - 0.5, hs.index(r.human) - 0.5), 1, 1, fill=False, ec=INK,
                                   lw=1.2, ls="--"))
        h, m = PICKS[pid][d]
        ax.add_patch(Rectangle((ms.index(m) - 0.5, hs.index(h) - 0.5), 1, 1, fill=False, ec=INK, lw=2.2))
        ax.set_xticks(range(len(ms)), ms, fontsize=8, color=INK, rotation=30, ha="right")
        ax.set_yticks(range(len(hs)), hs, fontsize=8, color=INK)
        ax.tick_params(length=0)
        for sp_ in ax.spines.values():
            sp_.set_visible(False)
        ax.set_title(f"{pid}, {DIR_NAME[d]}: pick {s[s.best].kappa_bal.iloc[0]:.3f}", fontsize=9, color=INK)
fig.colorbar(im, ax=axes, shrink=0.4, label="Wasserstein κ_bal")
fig.suptitle(f"{W_RUN}: κ_bal per section pair (rows: human, columns: mouse; solid: pick, dashed: partner)",
             color=INK)
fig.savefig(OUT / f"{STEM}_wasserstein_kappa.png", dpi=150, bbox_inches="tight")
plt.show()

## The influence of the spatial term

Each score of the picked section pairs as a function of α: the Wasserstein coupling at α = 0 (open marker), FGW at each α of `ALPHAS`. One column per cross pair, one row per score (shared y within a row); the colour is the query's species (blue: human query, b_stack; orange: mouse query, a_stack).

In [ ]:
def curve(pid, d, col):
    """(alphas, values) of one direction's picked section pair: Wasserstein at alpha 0, then FGW at each alpha."""
    w = W[(W.pair == pid) & (W.direction == d) & W.best][col].iloc[0]
    f = F[(F.pair == pid) & (F.direction == d)].sort_values("alpha")
    return np.r_[0.0, f.alpha.to_numpy()], np.r_[w, f[col].to_numpy()]


DIR_COLOR = {"b_stack": HUMAN, "a_stack": MOUSE}
DIR_SHORT = {"b_stack": "human → mouse", "a_stack": "mouse → human"}


def style(ax):
    ax.grid(color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    ax.tick_params(colors=MUTED, length=0, labelsize=8)
    for sp_ in ax.spines.values():
        sp_.set_visible(False)
    ax.set_xticks(np.r_[0.0, ALPHAS], ["W"] + [f"{a:g}" for a in ALPHAS])
    ax.set_xlim(-0.05, max(ALPHAS) + 0.05)


fig, axes = plt.subplots(len(METRICS), len(RUN_PAIRS), squeeze=False, sharex=True, sharey="row",
                         figsize=(2.6 * len(RUN_PAIRS) + 1, 2.3 * len(METRICS) + 1), layout="constrained")
for i, (col, title) in enumerate(METRICS.items()):
    for j, pid in enumerate(RUN_PAIRS):
        ax = axes[i, j]
        for d in DIRS:
            x, y = curve(pid, d, col)
            ax.plot(x, y, color=DIR_COLOR[d], lw=1.5, zorder=2)
            ax.plot(x[1:], y[1:], "o", color=DIR_COLOR[d], ms=6, mec=SURFACE, mew=1.2, zorder=3)
            ax.plot(x[:1], y[:1], "o", color=SURFACE, ms=6, mec=DIR_COLOR[d], mew=1.5, zorder=3)
        style(ax)
        if i == 0:
            ax.set_title(pid, color=INK, fontsize=10)
        if j == 0:
            ax.set_ylabel(title, color=INK)
        if i == len(METRICS) - 1:
            ax.set_xlabel("α (weight of the GW term)", color=MUTED, fontsize=8)
handles = [Line2D([], [], color=DIR_COLOR[d], lw=1.5, marker="o", ms=6, mec=SURFACE, label=DIR_SHORT[d])
           for d in DIRS]
handles.append(Line2D([], [], ls="none", marker="o", ms=6, color=SURFACE, mec=MUTED, mew=1.5,
                      label="Wasserstein (α = 0)"))
fig.legend(handles=handles, loc="outside lower center", ncol=3, frameon=False, fontsize=9, labelcolor=INK)
fig.suptitle(f"Picked section pairs: score vs. α ({MARG}, MAX_OBS {MAX_OBS}, seed {SEED})", color=INK)
fig.savefig(OUT / f"{STEM}_alpha_sweep.png", dpi=150, bbox_inches="tight")
plt.show()

### Change from Wasserstein, all pairs together

The same curves minus their Wasserstein value, so every line starts at 0 and the pairs are compared on one axis: one panel per score and direction, one line per cross pair (labelled at its end).

In [ ]:
PAIR_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
if len(RUN_PAIRS) > len(PAIR_COLORS):
    raise ValueError(f"at most {len(PAIR_COLORS)} pairs in this plot")
PCOL = {pid: PAIR_COLORS[list(SERIES.pairs).index(pid) % len(PAIR_COLORS)] for pid in RUN_PAIRS}  # fixed per pair

fig, axes = plt.subplots(len(METRICS), len(DIRS), squeeze=False, sharex=True, sharey="row",
                         figsize=(5.2 * len(DIRS) + 0.6, 2.4 * len(METRICS) + 1), layout="constrained")
for i, (col, title) in enumerate(METRICS.items()):
    for j, d in enumerate(DIRS):
        ax = axes[i, j]
        ax.axhline(0, color=MUTED, lw=1, zorder=1)
        for pid in RUN_PAIRS:
            x, y = curve(pid, d, col)
            ax.plot(x, y - y[0], color=PCOL[pid], lw=1.5, marker="o", ms=5, mec=SURFACE, mew=1, zorder=2)
            ax.annotate(pid, (x[-1], y[-1] - y[0]), xytext=(6, 0), textcoords="offset points", va="center",
                        fontsize=8, color=INK)
        style(ax)
        ax.set_xlim(-0.05, max(ALPHAS) + 0.15)
        if i == 0:
            ax.set_title(DIR_NAME[d], color=INK, fontsize=10)
        if j == 0:
            ax.set_ylabel(f"Δ {title}", color=INK)
        if i == len(METRICS) - 1:
            ax.set_xlabel("α (weight of the GW term)", color=MUTED, fontsize=8)
fig.legend(handles=[Line2D([], [], color=PCOL[pid], lw=1.5, marker="o", ms=5, label=pid) for pid in RUN_PAIRS],
           loc="outside lower center", ncol=len(RUN_PAIRS), frameon=False, fontsize=9, labelcolor=INK)
fig.suptitle("FGW minus Wasserstein on the picked section pairs", color=INK)
fig.savefig(OUT / f"{STEM}_alpha_sweep_delta.png", dpi=150, bbox_inches="tight")
plt.show()

### The numbers

Per pair and direction, the picked section pair (query → target, `partner`: the target is the query's partner section) and each score of Wasserstein (`W`) and FGW at each α.

In [ ]:
picked = W[W.best].set_index(["pair", "direction"])
for col, title in METRICS.items():
    t = F.pivot_table(index=["pair", "direction"], columns="alpha", values=col)
    t.columns = [f"α {a:g}" for a in t.columns]
    t.insert(0, "W", picked[col])
    t.insert(0, "partner", picked.partner)
    t.insert(0, "query → target", picked["query"] + " → " + picked["target"])
    print(f"{title}\n{t.round(3).to_string()}\n")

### Check against `label_alignment.ipynb`

Where `label_alignment.ipynb` has scored a pair at the same seed, `MAX_OBS` and marginals (`<root>/diag/label_alignment/<PAIR>_seed<SEED>_n<RES>_<MARG>.csv`), its rows for the same run, direction and section pair must agree with these (same cost, solver and ε), provided both ran at the same `INNER_TOL` and its FGW run used an α of `ALPHAS`.

In [ ]:
ORIG = root / "diag" / "label_alignment"
for pid in RUN_PAIRS:
    path = ORIG / f"{pid}_seed{SEED}_n{RES}_{MARG}.csv"
    if not path.exists():
        print(f"{pid}: no {path.name}, skipped")
        continue
    keys = ["run", "direction", "query", "target"]
    both = scores[scores.pair == pid].merge(pd.read_csv(path), on=keys, suffixes=("", "_orig"))
    for run, g in both.groupby("run", sort=False):
        diffs = ", ".join(f"{c} {np.nanmax(np.abs(g[c] - g[c + '_orig'])):.1e}" for c in METRICS)
        print(f"{pid} {run}: {len(g)} rows in common, max |difference| {diffs}")

## How to read the results

- **Score against α:** α weighs space against the labels. With a cross cost this close to perfect, the GW term mostly pulls the coupling away from the labels, so κ_bal is expected to stay near or below its Wasserstein value (Wasserstein minimizes the label cost, not κ_bal, so small gains are possible). A flat curve means the two sections' geometries agree with their organ layout; the α where it starts to fall is how much spatial weight the pair tolerates. A steep fall means space and labels disagree (different cut angle, depth or orientation).
- **The pick:** the section pair with the highest Wasserstein κ_bal is the one whose organs overlap best, not necessarily the partner section; the heatmaps show how clear the pick is.
- **The two directions:** under balanced marginals a section pair picked by both directions shares one coupling, and the two curves differ only in which side is scored.
- **Gene r:** no genes enter a label coupling, so gene r is the non-circular score here; it favours spread-out couplings (doc, section 6), so read it together with κ_bal.